# COS801 RQ7.1 — Model compression benchmark
This notebook runs the same leakage-audited split through FP32 baselines, INT8 quantisation, structured pruning, and knowledge distillation. Start with smoke mode; only full-mode results belong in the report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change these paths once for your Drive layout.
PROJECT_ROOT = '/content/drive/MyDrive/COS801_Project/rq7.1-model-compression'
DATA_DIR = '/content/drive/MyDrive/COS801_data/plantvillage'
OUTPUT_DIR = '/content/drive/MyDrive/COS801_results/rq7.1'
SMOKE_MODE = True
CONFIG = f'{PROJECT_ROOT}/configs/{"smoke" if SMOKE_MODE else "full"}.json'

In [ ]:
!pip -q install onnx onnxruntime torch-pruning scikit-learn
%cd $PROJECT_ROOT

In [ ]:
# Keep the shared configuration reproducible while supplying machine-specific paths.
import json
from pathlib import Path
cfg = json.loads(Path(CONFIG).read_text())
cfg['data_dir'] = DATA_DIR
cfg['output_dir'] = OUTPUT_DIR + ('/smoke' if SMOKE_MODE else '/full')
RUNTIME_CONFIG = '/content/rq71_runtime_config.json'
Path(RUNTIME_CONFIG).write_text(json.dumps(cfg, indent=2))
cfg

## 1. EDA and leakage gate

In [ ]:
!python run_experiment.py eda --config $RUNTIME_CONFIG

## 2. FP32 baselines

In [ ]:
!python run_experiment.py train --config $RUNTIME_CONFIG --model mobilenet_v3_small
!python run_experiment.py train --config $RUNTIME_CONFIG --model efficientnet_b0

## 3. Compression experiments
Select the better validation baseline as the teacher/checkpoint before a full run.

In [ ]:
TEACHER = f"{cfg['output_dir']}/mobilenet_v3_small/baseline.pt"
!python run_experiment.py quantize --config $RUNTIME_CONFIG --checkpoint $TEACHER
!python run_experiment.py prune --config $RUNTIME_CONFIG --checkpoint $TEACHER
!python run_experiment.py distill --config $RUNTIME_CONFIG --checkpoint $TEACHER

## 4. Results table

In [ ]:
import pandas as pd
results = pd.read_csv(f"{cfg['output_dir']}/metrics.csv")
results.sort_values(['accuracy', 'model_size_mb'], ascending=[False, True])